In [126]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

In [127]:
traidingFee= 0.0010 # https://bitflyer.com/en-eu/commission

In [128]:
df_sentiment = pd.read_parquet("bitcoin_userpost_sentiment.parquet")

In [129]:
df_sentiment.head(10)

,datetime,username,sentiment,sentiment_score
0,2021-01-01 23:59:58+00:00,WoodooProd,neutral,0.440227
1,2021-01-01 23:59:53+00:00,YSojourners,neutral,0.498390
2,2021-01-01 23:59:53+00:00,Andama_n,negative,0.438494
3,2021-01-01 23:59:51+00:00,YSojourners,negative,0.431339
4,2021-01-01 23:59:50+00:00,HunnerSa,neutral,0.442655
5,2021-01-01 23:59:49+00:00,WGeorgeCook,negative,0.523701
6,2021-01-01 23:59:49+00:00,TheCryptoKuber,negative,0.539488
7,2021-01-01 23:59:46+00:00,RDFIT4U,negative,0.420530
8,2021-01-01 23:59:46+00:00,HodlMagoo,negative,0.522886
9,2021-01-01 23:59:41+00:00,jessedo,negative,0.473934


In [130]:
len(df_sentiment)

15630309

In [131]:
df_sentiment["datetime"]= pd.to_datetime(df_sentiment["datetime"], utc=True)

In [132]:
sentiment_map = {"negative": 0, "neutral": 1, "positive": 2}
sentiment_map_invers= {0 : "negative", 1 : "neutral", 2 : "positive"}
df_sentiment["sentiment_val"] = df_sentiment["sentiment"].map(sentiment_map)
df_sentiment["weighted_val"] = df_sentiment["sentiment_val"] * df_sentiment["sentiment_score"]

In [133]:
print(df_sentiment['datetime'].min(), df_sentiment['datetime'].max())

2021-01-01 00:00:01+00:00 2021-12-31 23:59:59+00:00


In [134]:
df_btc= pd.read_csv("btcusd_1-min_data.csv")

In [135]:
df_btc['datetime'] = pd.to_datetime(df_btc['Timestamp'], unit='s', utc=True)
df_btc = df_btc[(df_btc['datetime'] >= df_sentiment['datetime'].min()) & (df_btc['datetime'] <= df_sentiment['datetime'].max())]
df_btc = df_btc.set_index('datetime')
df_sentiment= df_sentiment.set_index('datetime')

In [136]:
df_btc_1h= df_btc.resample('1h').agg({
    'Open':'first',
    'High':'max',
    'Low':'min',
    'Close':'last',
    'Volume':'sum'
})

In [137]:
df_btc_1h["return"]= df_btc_1h["Close"].pct_change()
df_btc_1h["return_class"] = sentiment_map["neutral"]

df_btc_1h.loc[df_btc_1h["return"] > traidingFee*2, "return_class"] = sentiment_map["positive"]
df_btc_1h.loc[df_btc_1h["return"] < -traidingFee*2, "return_class"] = sentiment_map["negative"]

In [138]:
df_sentiment_1h= df_sentiment.resample('1h').agg(
    avg_score=("sentiment_score", "mean"),
    n_positive=("sentiment", lambda x: (x=="positive").sum()),
    n_negative=("sentiment", lambda x: (x=="negative").sum()),
    total_tweets=("sentiment", "count"),
    net_sentiment=("sentiment_val", "sum"),
    sentiment_median_val=("sentiment_val", "median"),
    sentiment_mode=("sentiment", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    sentiment_mean=("sentiment_val", "mean"),  # einfacher Mittelwert codierter Sentiments
    weighted_val_sum=("weighted_val", "sum"),
    score_sum=("sentiment_score", "sum")
)

In [139]:
df_sentiment_1h["sentiment_median"] = df_sentiment_1h["sentiment_median_val"].round().map(sentiment_map_invers)
df_sentiment_1h["sentiment_weighted"] = (df_sentiment_1h["weighted_val_sum"] / df_sentiment_1h["score_sum"])
df_sentiment_1h.drop(columns=["weighted_val_sum", "score_sum","sentiment_median_val"], inplace=True)

In [140]:
df_btc_15min= df_btc.resample('15min').agg({
    'Open':'first',
    'High':'max',
    'Low':'min',
    'Close':'last',
    'Volume':'sum'
})

In [141]:
df_btc_15min["return"]= df_btc_15min["Close"].pct_change()
df_btc_15min["return_class"] = sentiment_map["neutral"]

df_btc_15min.loc[df_btc_15min["return"] > traidingFee*2, "return_class"] = sentiment_map["positive"]
df_btc_15min.loc[df_btc_15min["return"] < -traidingFee*2, "return_class"] = sentiment_map["negative"]

In [142]:
df_sentiment_15min= df_sentiment.resample('15min').agg(
    avg_score=("sentiment_score", "mean"),
    n_positive=("sentiment", lambda x: (x=="positive").sum()),
    n_negative=("sentiment", lambda x: (x=="negative").sum()),
    total_tweets=("sentiment", "count"),
    net_sentiment=("sentiment_val", "sum"),
    sentiment_median_val=("sentiment_val", "median"),
    sentiment_mode=("sentiment", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    sentiment_mean=("sentiment_val", "mean"),  # einfacher Mittelwert codierter Sentiments
    weighted_val_sum=("weighted_val", "sum"),
    score_sum=("sentiment_score", "sum")
)

In [143]:
df_sentiment_15min["sentiment_median"] = df_sentiment_15min["sentiment_median_val"].round().map(sentiment_map_invers)
df_sentiment_15min["sentiment_weighted"] = (df_sentiment_15min["weighted_val_sum"] / df_sentiment_15min["score_sum"])
df_sentiment_15min.drop(columns=["weighted_val_sum", "score_sum","sentiment_median_val"], inplace=True)

In [144]:
df_1h = df_btc_1h.join(df_sentiment_1h, how="inner")

In [145]:
df_1h.to_parquet("1h_btc_sentiment.parquet")

In [146]:
df_15min = df_btc_15min.join(df_sentiment_15min, how="inner")

In [147]:
df_15min.to_parquet("15min_btc_sentiment.parquet")

In [148]:
data= np.load("user_cluster_dict_total.npz", allow_pickle=True)
user_cluster_dict_total = data["user_cluster_dict"].item()

In [149]:
df_sentiment["cluster"] = df_sentiment["username"].map(user_cluster_dict_total)

In [150]:
agg_long_1h = (
    df_sentiment.groupby("cluster")
      .resample("1h")
      .agg(
          avg_score=("sentiment_score", "mean"),
          n_positive=("sentiment", lambda x: (x == "positive").sum()),
          n_negative=("sentiment", lambda x: (x == "negative").sum()),
          total_tweets=("sentiment", "count"),
          net_sentiment=("sentiment_val", "sum"),
          sentiment_median_val=("sentiment_val", "median"),
          sentiment_mode=("sentiment", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
          sentiment_mean=("sentiment_val", "mean"),
          weighted_val_sum=("weighted_val", "sum"),
          score_sum=("sentiment_score", "sum"),
      )
      .reset_index()
)

C:\Users\johan\AppData\Local\Temp\ipykernel_15748\3184311705.py:2: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sentiment.groupby("cluster")


In [151]:
agg_long_1h["sentiment_median"] = agg_long_1h["sentiment_median_val"].round().map(sentiment_map_invers)
agg_long_1h["sentiment_weighted"] = np.where(
    agg_long_1h["score_sum"].ne(0),
    agg_long_1h["weighted_val_sum"] / agg_long_1h["score_sum"],
    np.nan
)

In [152]:
agg_long_1h = agg_long_1h.drop(columns=["weighted_val_sum", "score_sum", "sentiment_median_val"])

In [153]:
df_sentiment_cluster_1h = (
    agg_long_1h
      .set_index(["datetime", "cluster"])
      .sort_index()
      .unstack("cluster")
)

In [154]:
df_sentiment_cluster_1h.columns = [f"{feat}_c{cl}" for feat, cl in df_sentiment_cluster_1h.columns]

In [155]:
agg_long_15min = (
    df_sentiment.groupby("cluster")
      .resample("15min")
      .agg(
          avg_score=("sentiment_score", "mean"),
          n_positive=("sentiment", lambda x: (x == "positive").sum()),
          n_negative=("sentiment", lambda x: (x == "negative").sum()),
          total_tweets=("sentiment", "count"),
          net_sentiment=("sentiment_val", "sum"),
          sentiment_median_val=("sentiment_val", "median"),
          sentiment_mode=("sentiment", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
          sentiment_mean=("sentiment_val", "mean"),
          weighted_val_sum=("weighted_val", "sum"),
          score_sum=("sentiment_score", "sum"),
      )
      .reset_index()
)

C:\Users\johan\AppData\Local\Temp\ipykernel_15748\679664626.py:2: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sentiment.groupby("cluster")


In [156]:
agg_long_15min["sentiment_median"] = agg_long_15min["sentiment_median_val"].round().map(sentiment_map_invers)
agg_long_15min["sentiment_weighted"] = np.where(
    agg_long_15min["score_sum"].ne(0),
    agg_long_15min["weighted_val_sum"] / agg_long_15min["score_sum"],
    np.nan
)

In [157]:
agg_long_15min = agg_long_15min.drop(columns=["weighted_val_sum", "score_sum", "sentiment_median_val"])

In [158]:
df_sentiment_cluster_15min = (
    agg_long_15min
      .set_index(["datetime", "cluster"])
      .sort_index()
      .unstack("cluster")
)

In [159]:
df_sentiment_cluster_15min.columns = [f"{feat}_c{cl}" for feat, cl in df_sentiment_cluster_15min.columns]

In [160]:
df_cluster_1h = df_btc_1h.join(df_sentiment_cluster_1h, how="inner")

In [161]:
df_cluster_1h.to_parquet("1h_btc_sentiment_cluster.parquet")

In [162]:
df_cluster_15min = df_btc_15min.join(df_sentiment_cluster_15min, how="inner")

In [163]:
df_cluster_15min.to_parquet("15min_btc_sentiment_cluster.parquet")